**Code Vision OCR Submodel**

**Importing Dependencies**

In [ ]:
!pip install Levenshtein
import Levenshtein

In [ ]:
!pip install --upgrade google-cloud-vision
from google.cloud import vision

In [ ]:
import io
import os
import pandas as pd
from PIL import Image, ImageEnhance
import re
import cv2
import numpy as np

In [ ]:
!unzip /content/screenshots-gen-ai.zip

In [ ]:
!unzip /content/gen-ai-test.zip
!unzip /content/gen-ai-test-cs.zip

In [ ]:
def extract_code_from_image(image_path):
  os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = '/content/ocr-for-gen-ai-fac0fbc83a73.json'
  client = vision.ImageAnnotatorClient()

  with io.open(image_path,'rb') as image_file:
    content = image_file.read()

  image = vision.Image(content=content)
  response = client.document_text_detection(image=image)

  text = response.full_text_annotation.text
  return text

**Preprocessing Functions**

In [ ]:
def binarize_image(image_path):
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    binary = cv2.adaptiveThreshold(img, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                   cv2.THRESH_BINARY, 11, 2)
    #cv2.imwrite("binarized.png", binary)
    return "binarized.png"

def preprocess_image(input_path, output_path="processed.png"):
    img = Image.open(input_path).convert("L")  # grayscale
    enhancer = ImageEnhance.Contrast(img)
    enhanced_image = enhancer.enhance(2.0)
    enhanced_image.save(output_path)
    return output_path

**NLD**

In [ ]:
def normalized_levenshtein(s1, s2):

    dist = Levenshtein.distance(s1, s2)
    max_len = max(len(s1), len(s2))

    return dist / max_len if max_len > 0 else 0.0

def log_nld(index, nld, log_file="log.txt"):
    with open(log_file, "a") as f:
        f.write(f"Image {index} | NLD: {nld:.4f}\n")

def normalize(code):
    return ''.join(code.strip().split())

In [ ]:
def run(language,clean):
    df = pd.read_csv('ocr_data.csv') # ground truth code from csv dataset

    image_folder = f"/content/screenshots-gen-ai/{language}"

    nld_iter = []
    exact,accurate = 0,0

    for i in range(100):
        image_path = os.path.join(image_folder,f'{i}.png')
        #print(f"Using image {i}: {image_path}")

        if clean:
            cleaned_image = preprocess_image(image_path) # preprocessing
            #cleaned_image = binarize_image(image_path)
            code_text = extract_code_from_image(cleaned_image) #code extraction
        else:
            code_text = extract_code_from_image(image_path)
        cleaned_code = re.sub(r"\s+", "", code_text) #no whitespace

        #groundTruth code from csv
        if language == 'python':
            code_gt = df.iloc[i]['Code'] # for python
        elif language == 'java':
            code_gt = df.iloc[100+i]['Code'] # for java

        code_gt = code_gt.replace(r"\n", "\n")
        code_groundTruth = re.sub(r"\s+", "", code_gt) #no whitespace

        nld = normalized_levenshtein(normalize(cleaned_code),normalize(code_groundTruth))
        if nld >= 0.5:
            print(f"Image {i}.png → NLD: {nld:.4f}")
            print('code:',cleaned_code)
            print('groundTruth:',code_groundTruth)
        elif nld <= 0.05:
            exact += 1
            accurate += 1
        elif nld <= 0.1:
            accurate += 1
        nld_iter.append(nld)
        log_nld(i,nld=nld)

    avg = sum(nld_iter)/len(nld_iter)
    return nld_iter, avg, exact, accurate

In [ ]:
nld_iter, avg, exact, accurate = run('java',clean=True)
print(nld_iter)
print('Average NLD:', sum(nld_iter)/len(nld_iter))
print('Exact:', exact)
print('Accurate:', accurate)

In [ ]:
def run_specific(language,clean):
    df = pd.read_csv('ocr_data.csv') # ground truth code from csv dataset

    image_folder = f"/content/screenshots-gen-ai/{language}"

    nld_iter = []
    exact,accurate = 0,0

    i = np.random.randint(0,99)

    image_path = os.path.join(image_folder,f'{i}.png')
    #print(f"Using image {i}: {image_path}")

    if clean:
        cleaned_image = preprocess_image(image_path) # preprocessing
            #cleaned_image = binarize_image(image_path)
        code_text = extract_code_from_image(cleaned_image) #code extraction
    else:
        code_text = extract_code_from_image(image_path)
    cleaned_code = re.sub(r"\s+", "", code_text) #no whitespace

    return normalize(cleaned_code)

In [ ]:
print(run_specific('java',clean=True))

In [ ]:
def runJavaTest(clean):

    image_folder = f"/content/gen-ai-test"

    javaCodeTestList = []
    exact,accurate = 0,0

    for i in range(20):
        image_path = os.path.join(image_folder,f'{i}.png')
        #print(f"Using image {i}: {image_path}")

        if clean:
            cleaned_image = preprocess_image(image_path) # preprocessing
            #cleaned_image = binarize_image(image_path)
            code_text = extract_code_from_image(cleaned_image) #code extraction
        else:
            code_text = extract_code_from_image(image_path)
        cleaned_code = re.sub(r"\s+", "", code_text) #no whitespace
        javaCodeTestList.append(normalize(cleaned_code))

    return javaCodeTestList

In [ ]:
javaCodeTestList = runJavaTest(clean=True)
print(javaCodeTestList)

In [ ]:
def runCSTest(clean):

    image_folder = f"/content/gen-ai-test-cs"

    csCodeTestList = []
    exact,accurate = 0,0

    for i in range(20):
        image_path = os.path.join(image_folder,f'{i}.png')
        #print(f"Using image {i}: {image_path}")

        if clean:
            cleaned_image = preprocess_image(image_path) # preprocessing
            #cleaned_image = binarize_image(image_path)
            code_text = extract_code_from_image(cleaned_image) #code extraction
        else:
            code_text = extract_code_from_image(image_path)
        cleaned_code = re.sub(r"\s+", "", code_text) #no whitespace
        csCodeTestList.append(normalize(cleaned_code))

    return csCodeTestList

In [ ]:
csCodeTestList = runCSTest(clean=True)
print(csCodeTestList)

**Code Translation Submodel**

**Installing Library Dependencies**

In [ ]:
!pip install datasets
!pip install evaluate

**Importing Dataset**

In [ ]:
from datasets import load_dataset

dataset = load_dataset("google/code_x_glue_cc_code_to_code_trans")
print(dataset.column_names)
print(dataset['train'][0])

**Baseline Testing & Evaluation** \\
We will be testing two baseline models: CodeT5-Small and CodeT5-Base. For each model, we will be evaluating the model's performance on Java to C# translations and also C# to Java. The functions below define the model loading and evaluation process using the *code_x_glue* test dataset.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

def model_loader(model):
  if model=='base':
    model_name = "Salesforce/codet5-base"
  elif model=='small':
    model_name = "Salesforce/codet5-small"
  tokenizer = AutoTokenizer.from_pretrained(model_name)
  model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
  device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
  model.to(device)
  return tokenizer, model, device

In [ ]:
import evaluate
def javaToCSharp(tokenizer, model, device):
  predictions = []
  references = []

  print("Generating Java to C# translations ...")
  for example in dataset['test']:
      source_code = example["java"]
      reference_code = example["cs"]

      prompt = f"Translate Java to C#: {source_code}"

      inputs = tokenizer(prompt, return_tensors="pt", padding="max_length",
                        truncation=True, max_length=512)
      inputs = {k: v.to(device) for k, v in inputs.items()}

      outputs = model.generate(**inputs, max_length=512)
      translated_code = tokenizer.decode(outputs[0], skip_special_tokens=True)

      predictions.append(translated_code)
      references.append(reference_code)

  bleu_metric = evaluate.load("bleu")

  results = bleu_metric.compute(predictions=predictions, references=references)
  print("BLEU Score:", results["bleu"])

  exact_matches = 0
  for pred, ref in zip(predictions, references):
      if pred.strip() == ref.strip():
          exact_matches += 1

  exact_match_score = exact_matches / len(predictions)
  print("Exact Match Score:", exact_match_score)

def cSharpToJava(tokenizer, model, device):
  predictions = []
  references = []

  print("Generating C# to Java translations ...")
  for example in dataset['test']:
      source_code = example["cs"]
      reference_code = example["java"]

      prompt = f"Translate C# to Java: {source_code}"

      inputs = tokenizer(prompt, return_tensors="pt", padding="max_length",
                        truncation=True, max_length=512)
      inputs = {k: v.to(device) for k, v in inputs.items()}

      outputs = model.generate(**inputs, max_length=512)
      translated_code = tokenizer.decode(outputs[0], skip_special_tokens=True)

      predictions.append(translated_code)
      references.append(reference_code)

  bleu_metric = evaluate.load("bleu")

  results = bleu_metric.compute(predictions=predictions, references=references)
  print("BLEU Score:", results["bleu"])

  exact_matches = 0
  for pred, ref in zip(predictions, references):
      if pred.strip() == ref.strip():
          exact_matches += 1

  exact_match_score = exact_matches / len(predictions)
  print("Exact Match Score:", exact_match_score)

(i) CodeT5-Base Java → C#

In [ ]:
tokenizer, model, device = model_loader('base')
javaToCSharp(tokenizer, model, device)
cSharpToJava(tokenizer, model, device)

(iii) CodeT5-Small Java → C# + C# → Java

In [ ]:
# 4 min 38s run time on T4
tokenizer, model, device = model_loader('small')
javaToCSharp(tokenizer, model, device)
cSharpToJava(tokenizer, model, device)

**Fine Tuning Methods** \\
As seen above, both CodeT5 models perform poorly without any fine-tuning as the original model was built around several tasks across multiple language (ie. code generation, translation, etc.). In order to be effective for our objective, we explore the following fine-tuning methods: Top-K Layer Fine-Tune and Low Rank Adaptation (LoRA).

Top-K Layer Fine-Tune

In [ ]:
model_name = "Salesforce/codet5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# layer freeze
for param in model.parameters():
    param.requires_grad = False

K = 8  # adjust for number of layers

# layer unfreeze
for layer in model.encoder.block[-K:]:
    for param in layer.parameters():
        param.requires_grad = True
for layer in model.decoder.block[-K:]:
    for param in layer.parameters():
        param.requires_grad = True
for param in model.shared.parameters():
    param.requires_grad = True
for param in model.lm_head.parameters():
    param.requires_grad = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

In [ ]:
def preprocess(example):
    # for Java to C#
    #inputs = ["translate java to CSharp: " + src for src in example["java"]]
    #targets = example["cs"]

    # for C# to Java
    inputs = ["translate CSharp to java: " + src for src in example["cs"]]
    targets = example["java"]

    model_inputs = tokenizer(
        inputs,
        truncation=True,
        padding="max_length",
        max_length=256
    )

    labels = tokenizer(
        targets,
        truncation=True,
        padding="max_length",
        max_length=256
    ).input_ids

    model_inputs["labels"] = labels
    return model_inputs

In [ ]:
tokenized_dataset = dataset.map(preprocess, batched=True)

In [ ]:
### Top K training
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq

training_args = Seq2SeqTrainingArguments(
    output_dir=f"./codet5-topk-ft{K}",
    run_name=f"./codet5-topk-ft",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=5e-4, #original 5e-5
    num_train_epochs=3,
    logging_dir="./logs",
    fp16=True,
    save_steps=500,
    logging_steps=100,
    eval_steps=500,
    save_total_limit=2
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    tokenizer=tokenizer,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
)

trainer.train()
trainer.evaluate()

In [ ]:
def translate_example(example):
    # Java to CSharp
    #source_code = example['java']
    #input_text = "translate java to CSharp: " + source_code

    # CSharp to Java
    source_code = example['cs']
    input_text = "translate CSharp to java: " + source_code

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        padding="max_length",
        truncation=True,
        max_length=512
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}
    outputs = model.generate(**inputs, max_length=512)
    translated_code = tokenizer.decode(outputs[0], skip_special_tokens=True)

    example['predicted'] = translated_code
    return example

In [ ]:
def translate_specific_code(codeString):
    # Java to CSharp
    #input_text = "translate java to CSharp: " + codeString

    # CSharp to Java
    input_text = "translate CSharp to java: " + codeString

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        padding="max_length",
        truncation=True,
        max_length=512
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}
    outputs = model.generate(**inputs, max_length=512)
    translated_code = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return translated_code

In [ ]:
#java code from ocr
java_code = run_specific('java',clean=True)
print(java_code)
translation = translate_specific_code(java_code)
print(translation)

**Java → C# Full Pipeline Test**

In [ ]:
def translationJavaTest(codeList):
  translationList = []
  for i in range(len(codeList)):
      sourceCodeString = codeList[i]
      translation = translate_specific_code(sourceCodeString)
      translationList.append(translation)
  return translationList

In [ ]:
javaCodeTestList = runJavaTest(clean=True)
refs = translationJavaTest(javaCodeTestList)
preds = pd.read_csv('javaFinalTest.csv')
preds = preds.iloc[:,0].tolist()
print(preds,refs)
exact_matches = sum(p.strip() == r.strip() for p, r in zip(preds, refs))
em_score = exact_matches / len(preds) * 100

print(f"Exact Match: {em_score:.2f}%")

bleu = evaluate.load("bleu")
wrapped_refs = [[ref] for ref in refs]
bleu_score = bleu.compute(predictions=preds, references=wrapped_refs)
print("BLEU:", bleu_score["bleu"])

**C# → Java Full Pipeline Test**

In [ ]:
def translationCSTest(codeList):
  translationList = []
  for i in range(len(codeList)):
      sourceCodeString = codeList[i]
      translation = translate_specific_code(sourceCodeString)
      translationList.append(translation)
  return translationList

In [ ]:
csCodeTestList = runCSTest(clean=True)
refs = translationCSTest(csCodeTestList)
preds = pd.read_csv('csFinalTest.csv')
preds = preds.iloc[:,0].tolist()
print(preds,refs)
exact_matches = sum(p.strip() == r.strip() for p, r in zip(preds, refs))
em_score = exact_matches / len(preds) * 100

print(f"Exact Match: {em_score:.2f}%")

bleu = evaluate.load("bleu")
wrapped_refs = [[ref] for ref in refs]
bleu_score = bleu.compute(predictions=preds, references=wrapped_refs)
print("BLEU:", bleu_score["bleu"])

In [ ]:
def batch_translate_examples(examples):
    #Java to CSharp
    inputs = ["translate java to CSharp: " + src for src in examples["java"]]

    #CSharp to Java
    #inputs = ["translate CSharp to java: " + src for src in examples["cs"]]

    tokenized_inputs = tokenizer(
        inputs,
        return_tensors="pt",
        padding="max_length",
        truncation=True,
        max_length=512
    ).to(device)

    # predictions
    outputs = model.generate(
        **tokenized_inputs,
        max_length=512,
        num_beams=5,
    )

    # outputs to strings
    predictions = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    return {"predicted": predictions}

In [ ]:
translated_test_dataset = dataset["test"].map(batch_translate_examples, batched=True, batch_size=8)

In [ ]:
preds = translated_test_dataset["predicted"]
refs = translated_test_dataset["java"]

exact_matches = sum(p.strip() == r.strip() for p, r in zip(preds, refs))
em_score = exact_matches / len(preds) * 100

print(f"Exact Match: {em_score:.2f}%")

bleu = evaluate.load("bleu")
wrapped_refs = [[ref] for ref in refs]
bleu_score = bleu.compute(predictions=preds, references=wrapped_refs)
print("BLEU:", bleu_score["bleu"])

LoRA

In [ ]:
import math
import torch.nn as nn
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq
)
from transformers.models.t5.modeling_t5 import T5Attention

class LoRALinear(nn.Linear):
  def __init__(self, in_features, out_features, bias=True, device=None,
               dtype=None, lora_rank=8, lora_alpha=16, lora_dropout=0.1):
    super().__init__(in_features, out_features, bias=bias,
                         device=device, dtype=dtype)
    self.has_weights_merged = False
    if lora_rank > 0:
      self.lora_dropout = nn.Dropout(lora_dropout)
      self.lora_scaling = lora_alpha / lora_rank
      self.lora_A = nn.Parameter(torch.empty(lora_rank, in_features,
                                              device=device, dtype=dtype))
      self.lora_B = nn.Parameter(torch.empty(out_features, lora_rank,
                                              device=device, dtype=dtype))
      self.lora_A.requires_grad = True
      self.lora_B.requires_grad = True
      self.reset_parameters()

  def is_lora(self):
    return hasattr(self, "lora_A")

  def reset_parameters(self):
    nn.Linear.reset_parameters(self)
    if self.is_lora():
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B)

  def forward(self, input):
    output = super().forward(input)
    if self.is_lora() and not self.has_weights_merged:
      dropped = self.lora_dropout(input)
      A = dropped @ self.lora_A.T
      BA = A @ self.lora_B.T
      output += self.lora_scaling * BA
    return output


  def train(self, mode=True):
    super().train(mode)
    if not self.is_lora():
      return self
    if mode:
      if self.has_weights_merged:
        self.weight.data -= self.lora_scaling * (self.lora_B @ self.lora_A)
        self.has_weights_merged = False
    else:
      if not self.has_weights_merged:
        self.weight.data += self.lora_scaling * (self.lora_B @ self.lora_A)
        self.has_weights_merged = True
    return self


  def eval(self):
    super().eval()
    if self.is_lora() and not self.has_weights_merged:
      self.weight.data += self.lora_scaling * (self.lora_B @ self.lora_A)
      self.has_weights_merged = True
    return self

In [ ]:
def replace_with_lora(model, rank=8, alpha=16, target=("q","v")):
  for module in model.modules():
    if isinstance(module, T5Attention):
      for component in target:
        lin = getattr(module, component)
        lora_lin = LoRALinear(lin.in_features, lin.out_features, bias=False,
                              lora_rank=rank, lora_alpha=alpha,
                              lora_dropout=0.1)
        lora_lin.weight.data = lin.weight.data.clone()
        setattr(module, component, lora_lin)

def freeze_non_lora(model):
  for name, parameter in model.named_parameters():
    if "lora_A" in name or "lora_B" in name:
      parameter.requires_grad = True

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("Salesforce/codet5-base")

def encode(datapoint):
  # Java to CSharp
  #model_inputs = tokenizer(datapoint["java"],truncation=True, max_length=512)
  #labels = tokenizer(datapoint["cs"],truncation=True, max_length=512).input_ids

  # CSharp to Java
  model_inputs = tokenizer(datapoint["cs"],truncation=True, max_length=512)
  labels = tokenizer(datapoint["java"],truncation=True, max_length=512).input_ids
  model_inputs["labels"] = labels
  return model_inputs

train_ds = dataset['train'].map(encode, batched=True, remove_columns=dataset['train'].column_names)

In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained("Salesforce/codet5-base")
replace_with_lora(model, rank=8, alpha=16)
freeze_non_lora(model)

collator = DataCollatorForSeq2Seq(tokenizer, model=model)
args = TrainingArguments(
    "codet5_lora",
    report_to="none",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=4e-4,
    num_train_epochs=3,
    fp16=torch.cuda.is_available(),
    logging_steps=100,
    save_steps=1000
)

Trainer(model=model,
        args=args,
        train_dataset= train_ds,
        data_collator=collator).train()

model.eval()

model.save_pretrained("codet5_lora_final")
tokenizer.save_pretrained("codet5_lora_final")

In [ ]:
import evaluate
def javaToCSharp_specificTranslation(tokenizer, model, device,codeString):

  print("Generating Java to C# translation ...")
  source_code = codeString
  prompt = f"Translate Java to C#: {source_code}"
  inputs = tokenizer(prompt, return_tensors="pt", padding="max_length",
                        truncation=True, max_length=512)
  inputs = {k: v.to(device) for k, v in inputs.items()}

  outputs = model.generate(**inputs, max_length=512)
  translated_code = tokenizer.decode(outputs[0], skip_special_tokens=True)

  return translated_code


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained("codet5_lora_final")
tokenizer = AutoTokenizer.from_pretrained("codet5_lora_final")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

def translationJavaTest(codeList):
  translationList = []
  for i in range(len(codeList)):
      sourceCodeString = codeList[i]
      translation = javaToCSharp_specificTranslation(tokenizer, model, device, sourceCodeString)
      translationList.append(translation)
  return translationList

javaCodeTestList = runJavaTest(clean=True)
refs = translationJavaTest(javaCodeTestList)
preds = pd.read_csv('javaFinalTest.csv')
preds = preds.iloc[:,0].tolist()
print(preds,refs)
exact_matches = sum(p.strip() == r.strip() for p, r in zip(preds, refs))
em_score = exact_matches / len(preds) * 100

print(f"Exact Match: {em_score:.2f}%")

bleu = evaluate.load("bleu")
wrapped_refs = [[ref] for ref in refs]
bleu_score = bleu.compute(predictions=preds, references=wrapped_refs)
print("BLEU:", bleu_score["bleu"])

In [ ]:
import evaluate
def csToJava_specificTranslation(tokenizer, model, device,codeString):

  print("Generating C# to Java translation ...")
  source_code = codeString
  prompt = f"Translate C# to Java: {source_code}"
  inputs = tokenizer(prompt, return_tensors="pt", padding="max_length",
                        truncation=True, max_length=512)
  inputs = {k: v.to(device) for k, v in inputs.items()}

  outputs = model.generate(**inputs, max_length=512)
  translated_code = tokenizer.decode(outputs[0], skip_special_tokens=True)

  return translated_code

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained("codet5_lora_final")
tokenizer = AutoTokenizer.from_pretrained("codet5_lora_final")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

def translationCSTest(codeList):
  translationList = []
  for i in range(len(codeList)):
      sourceCodeString = codeList[i]
      translation = csToJava_specificTranslation(tokenizer, model, device, sourceCodeString)
      translationList.append(translation)
  return translationList

csCodeTestList = runCSTest(clean=True)
refs = translationCSTest(csCodeTestList)
preds = pd.read_csv('csFinalTest.csv')
preds = preds.iloc[:,0].tolist()
print(preds,refs)
exact_matches = sum(p.strip() == r.strip() for p, r in zip(preds, refs))
em_score = exact_matches / len(preds) * 100

print(f"Exact Match: {em_score:.2f}%")

bleu = evaluate.load("bleu")
wrapped_refs = [[ref] for ref in refs]
bleu_score = bleu.compute(predictions=preds, references=wrapped_refs)
print("BLEU:", bleu_score["bleu"])

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained("codet5_lora_final")
tokenizer = AutoTokenizer.from_pretrained("codet5_lora_final")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

#javaToCSharp(tokenizer, model, device)
#cSharpToJava(tokenizer, model, device)

#java code from ocr
java_code = run_specific('java',clean=True)
print(java_code)
translation = javaToCSharp_specificTranslation(tokenizer,model,device,java_code)
print(translation)